# Predictive Maintenance 7-Week Student Starter Notebook
This notebook supports the 7-week Co-op project. Week 6 is focused on Evaluation and Optimization, and Week 7 is the final demo week.

## Week 1: Maintenance Scenario Thinking

In [ ]:
equipment_case = {
    'Air_Temp_C': 38,
    'Process_Temp_C': 49,
    'Torque_Nm': 58,
    'Vibration_mm_s': 6.1,
    'Tool_Wear_Min': 210
}
for key, value in equipment_case.items():
    print(key, ':', value)
# Use ChatGPT/Copilot to explain whether this case looks risky and why.


## Week 2: Python Fundamentals

In [ ]:
temperature = 38
vibration = 6.1
if temperature > 35 or vibration > 5:
    print('Potential risk detected')
else:
    print('Normal condition')


## Week 3: Load and Explore Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
file_path = 'Predictive_Maintenance_Training_Dataset_7_Week.xlsx'
df = pd.read_excel(file_path, sheet_name='Training_Dataset')
df.head()


In [ ]:
df.info()
df.describe()


## Week 4: Visualization and Power BI Dashboard

In [ ]:
df['Machine_Failure'].value_counts().plot(kind='bar')
plt.title('Machine Failure Distribution')
plt.xlabel('Machine Failure')
plt.ylabel('Records')
plt.show()


In [ ]:
df.groupby('Equipment_Type')['Machine_Failure'].mean().sort_values(ascending=False).plot(kind='bar')
plt.title('Failure Rate by Equipment Type')
plt.ylabel('Failure Rate')
plt.show()


### Power BI Task
Open the Excel dataset in Power BI Desktop. Load Training_Dataset. Build visuals for risk distribution, failure type breakdown, failure rate by equipment type, and key sensor insights.

## Week 5: Machine Learning Model

In [ ]:
features = ['Equipment_Type','Quality_Type','Air_Temp_C','Process_Temp_C','Rotational_Speed_RPM','Torque_Nm','Tool_Wear_Min','Vibration_mm_s','Pressure_bar']
target = 'Machine_Failure'
X = df[features]
y = df[target]
categorical_features = ['Equipment_Type','Quality_Type']
numeric_features = [c for c in features if c not in categorical_features]
preprocess = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features), ('num', 'passthrough', numeric_features)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
baseline = Pipeline([('preprocess', preprocess), ('classifier', DecisionTreeClassifier(random_state=42, class_weight='balanced'))])
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print('Baseline Accuracy:', accuracy_score(y_test, baseline_pred))
print(confusion_matrix(y_test, baseline_pred))


In [ ]:
model = Pipeline([('preprocess', preprocess), ('classifier', RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced'))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('Random Forest Accuracy:', accuracy_score(y_test, pred))
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))


## Week 6: Evaluation and Optimization

In [ ]:
# Try a second Random Forest version and compare results
optimized_model = Pipeline([('preprocess', preprocess), ('classifier', RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight='balanced'))])
optimized_model.fit(X_train, y_train)
opt_pred = optimized_model.predict(X_test)
print('Optimized Random Forest Accuracy:', accuracy_score(y_test, opt_pred))
print(classification_report(y_test, opt_pred))
print(confusion_matrix(y_test, opt_pred))


Write a short evaluation note:
- Which model performed better?
- What type of error is more risky in maintenance: false positive or false negative?
- What recommendation logic will you use in the final demo?

## Week 7: Final Colab Demo Input

In [ ]:
sample = pd.DataFrame([{
    'Equipment_Type': 'Pump',
    'Quality_Type': 'Low',
    'Air_Temp_C': 38.0,
    'Process_Temp_C': 49.0,
    'Rotational_Speed_RPM': 1320,
    'Torque_Nm': 58.0,
    'Tool_Wear_Min': 210,
    'Vibration_mm_s': 6.1,
    'Pressure_bar': 13.5
}])
prediction = optimized_model.predict(sample)[0]
probability = optimized_model.predict_proba(sample)[0][1]
print('Prediction:', 'Failure' if prediction == 1 else 'No Failure')
print('Failure Probability:', round(probability*100,1), '%')
if probability >= 0.7:
    print('Recommendation: Immediate inspection recommended.')
elif probability >= 0.4:
    print('Recommendation: Schedule preventive inspection.')
else:
    print('Recommendation: Continue normal monitoring.')
